## FEATURE ENGINEERING AND DATA PREPROCESSING

In [ ]:
import pandas as pd
df=pd.DataFrame({'dob':['1995-03-15','1988-11-02','2001-07-20']})
df['dob']=pd.to_datetime(df['dob'])
df['age']=2024-df['dob'].dt.year
df['birth_month']=df['dob'].dt.month
df['birth_day']=df['dob'].dt.day_name()
bins=[0,28,44,60,100]
labels=['genz','millineal','genx','alpha']
df['generation']=pd.cut(df['age'],bins=bins,labels=labels)
print(df[['age','birth_month','birth_day','generation']])

   age  birth_month  birth_day generation
0   29            3  Wednesday  millineal
1   36           11  Wednesday  millineal
2   23            7     Friday       genz


In [ ]:
from sklearn.preprocessing import LabelEncoder
import pandas as pd
df['purchased']=['yes','no','yes']
le=LabelEncoder()
df['purchased']=le.fit_transform(df['purchased'])
df['size']=['small','medium','large']
size_order={'small':0,'medium':2,'large':6}
df['size_enc']=df['size'].map(size_order)
print(df[['purchased','size','size_enc']])

   purchased    size  size_enc
0          1   small         0
1          0  medium         2
2          1   large         6


In [ ]:
import pandas as pd
df = pd.DataFrame({'city': ['Delhi', 'Mumbai', 'Chennai', 'Delhi']})
df_encoded = pd.get_dummies(df, columns=['city'], dtype=int)
print(df_encoded)

print("--------------")
from sklearn.preprocessing import OneHotEncoder
ohe = OneHotEncoder(sparse_output=False, drop='first')
encoded = ohe.fit_transform(df[['city']])

print(encoded)

   city_Chennai  city_Delhi  city_Mumbai
0             0           1            0
1             0           0            1
2             1           0            0
3             0           1            0
--------------
[[1. 0.]
 [0. 1.]
 [0. 0.]
 [1. 0.]]


In [ ]:
import pandas as pd
import numpy as np
df=pd.DataFrame({
    'feature_A':[1,2,np.nan,4,5],
    'feature_B':["x","y","z",np.nan,"w"],
     'feature_c':[10.5,np.nan,12.3,13.0,np.nan],
})
print("missing value count percolumn")
print(df.isnull().sum())
print('percentage missing per column:')
print(df.isnull().mean()*100)


missing value count percolumn
feature_A    1
feature_B    1
feature_c    2
dtype: int64
percentage missing per column:
feature_A    20.0
feature_B    20.0
feature_c    40.0
dtype: float64


In [ ]:
from sklearn.impute import SimpleImputer
import numpy as np
numeric_cols = ['feature_A', 'feature_c']
num_imputer = SimpleImputer(strategy='median')

df[numeric_cols] = num_imputer.fit_transform(df[numeric_cols])

categorical_cols = ['feature_B']
cat_imputer = SimpleImputer(strategy='most_frequent')

df[categorical_cols] = cat_imputer.fit_transform(df[categorical_cols])


print('Total missing values after imputation:', df.isnull().sum().sum())
print('\nDataFrame after imputation:')
print(df)

Total missing values after imputation: 0

DataFrame after imputation:
   feature_A feature_B  feature_c
0        1.0         x       10.5
1        2.0         y       12.3
2        3.0         z       12.3
3        4.0         w       13.0
4        5.0         w       12.3


In [ ]:
Q1 = df['feature_A'].quantile(0.25)
Q3 = df['feature_A'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR
df_clean_iqr = df[(df['feature_A'] >= lower) & (df['feature_A'] <= upper)]
print("DataFrame after IQR outlier removal on 'feature_A':")
print(df_clean_iqr)
print('\n')
from scipy import stats
z_scores = stats.zscore(df['feature_A'])
df_clean_zscore = df[abs(z_scores) < 3]
print("DataFrame after Z-score outlier removal on 'feature_A' (threshold 3):")
print(df_clean_zscore)

DataFrame after IQR outlier removal on 'feature_A':
   feature_A feature_B  feature_c
0        1.0         x       10.5
1        2.0         y       12.3
2        3.0         z       12.3
3        4.0         w       13.0
4        5.0         w       12.3


DataFrame after Z-score outlier removal on 'feature_A' (threshold 3):
   feature_A feature_B  feature_c
0        1.0         x       10.5
1        2.0         y       12.3
2        3.0         z       12.3
3        4.0         w       13.0
4        5.0         w       12.3


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
X = df[['feature_A', 'feature_c']]
y = (df['feature_A'] > df['feature_A'].median()).astype(int)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
pipe = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(random_state=42)),
])

pipe.fit(X_train, y_train)
print('Accuracy:', pipe.score(X_test, y_test))
X_new = pd.DataFrame({
    'feature_A': [6.0, np.nan, 8.0],
    'feature_c': [15.0, 16.0, np.nan]
})
predictions = pipe.predict(X_new)
print('Predictions for new data:', predictions)

Accuracy: 1.0
Predictions for new data: [1 1 1]
